In [1]:
!pip install -q kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and extract the dataset
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:16<00:00, 136MB/s]



In [2]:
import os
import shutil

base_dir = 'plantvillage dataset/color'
unknown_dir = os.path.join(base_dir, 'Unknown_Crop')
os.makedirs(unknown_dir, exist_ok=True)

# Grab images from plants that ARE NOT tomatoes or potatoes
other_folders = [
    'Strawberry___healthy',
    'Corn_(maize)___healthy',
    'Apple___healthy',
    'Soybean___healthy'
]

for folder in other_folders:
    src_path = os.path.join(base_dir, folder)
    if os.path.exists(src_path):
        # Grab exactly 100 images from each to keep the dataset balanced
        images = os.listdir(src_path)[:100]
        for img in images:
            shutil.copy(os.path.join(src_path, img), os.path.join(unknown_dir, img))

print(f"Success! Created Unknown_Crop class with {len(os.listdir(unknown_dir))} images.")

Success! Created Unknown_Crop class with 400 images.


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

base_dir = 'plantvillage dataset/color'

# The exact 6 classes for high-accuracy training
focus_classes = [
    'Tomato___healthy',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Potato___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Unknown_Crop' # <--- The new garbage collector class
]
# Data Augmentation (only for training)
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2, # 80% train, 20% test/val
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Training Generator
train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    classes=focus_classes
)

# Validation/Testing Generator
val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    classes=focus_classes,
    shuffle=False # Crucial for accurate evaluation later
)

print("\nClass Index Mapping:")
print(train_gen.class_indices)

Found 5643 images belonging to 7 classes.
Found 1409 images belonging to 7 classes.

Class Index Mapping:
{'Tomato___healthy': 0, 'Tomato___Early_blight': 1, 'Tomato___Late_blight': 2, 'Potato___healthy': 3, 'Potato___Early_blight': 4, 'Potato___Late_blight': 5, 'Unknown_Crop': 6}


In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

# 1. Load Pre-trained Base
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # Freeze base model

# 2. Build Custom Head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(focus_classes), activation='softmax') # 6 output nodes
])

# 3. Compile and Train
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Starting Phase 1: Training custom layers...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Starting Phase 1: Training custom layers...
Epoch 1/10
 61/177 ━━━━━━━━━━━━━━━━━━━━ 10:05 5s/step - accuracy: 0.4677 - loss: 1.5869

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# 1. Unfreeze the base model
base_model.trainable = True

# 2. Keep the first 300 layers frozen to retain basic edge/color detection
fine_tune_at = 300
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# 3. Recompile with a 10x smaller learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 4. Prevent overfitting with Early Stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Starting Phase 2: Fine-tuning deeper layers...")
history_fine = model.fit(
    train_gen,
    epochs=20, # Will likely stop early due to the callback
    initial_epoch=history.epoch[-1],
    validation_data=val_gen,
    callbacks=[early_stop]
)

# Save the final masterpiece
model.save('sish_nightshade_model.h5')
print("Model successfully saved as 'sish_nightshade_model.h5'")

Starting Phase 2: Fine-tuning deeper layers...
Epoch 10/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 214s 862ms/step - accuracy: 0.8260 - loss: 0.5452 - val_accuracy: 0.9659 - val_loss: 0.1103
Epoch 11/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 98s 552ms/step - accuracy: 0.9238 - loss: 0.2212 - val_accuracy: 0.9659 - val_loss: 0.1035
Epoch 12/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 97s 546ms/step - accuracy: 0.9406 - loss: 0.1750 - val_accuracy: 0.9730 - val_loss: 0.0823
Epoch 13/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 98s 553ms/step - accuracy: 0.9498 - loss: 0.1479 - val_accuracy: 0.9808 - val_loss: 0.0655
Epoch 14/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 95s 535ms/step - accuracy: 0.9612 - loss: 0.1136 - val_accuracy: 0.9759 - val_loss: 0.0625
Epoch 15/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 95s 538ms/step - accuracy: 0.9642 - loss: 0.1024 - val_accuracy: 0.9737 - val_loss: 0.0754
Epoch 16/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 98s 551ms/step - accuracy: 0.9670 - loss: 0.0943 - val_accuracy: 0.9787 - val_loss: 0.0705
Epoch 17/20
177/177 ━━━━━━━━

Model successfully saved as 'sish_nightshade_model.h5'


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Get true labels and model predictions
val_gen.reset() # Reset generator to ensure alignment
predictions = model.predict(val_gen)
y_pred = np.argmax(predictions, axis=1)
y_true = val_gen.classes

# Get class names
class_names = list(val_gen.class_indices.keys())

# Print numerical report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Testing Data Confusion Matrix')
plt.ylabel('True Disease')
plt.xlabel('Predicted Disease')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

NameError: name 'model' is not defined

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
from google.colab import files

def upload_and_predict(model_path='sish_nightshade_model.h5'):
    """
    Creates an interactive upload button in Colab, displays the image,
    and predicts the crop disease with an Unknown Crop filter.
    """
    print("Please upload a crop leaf image (Tomato or Potato):")
    uploaded = files.upload()

    if not uploaded:
        print("No image uploaded.")
        return

    # Updated class mapping to exactly match your focus_classes list
    class_indices = {
        0: 'Tomato___healthy',
        1: 'Tomato___Early_blight',
        2: 'Tomato___Late_blight',
        3: 'Potato___healthy',
        4: 'Potato___Early_blight',
        5: 'Potato___Late_blight',
        6: 'Unknown_Crop'
    }

    print("Loading model...")
    model = tf.keras.models.load_model(model_path)

    for file_name in uploaded.keys():
        print(f"\n--- Analyzing: {file_name} ---")

        img_display = image.load_img(file_name)
        plt.figure(figsize=(4, 4))
        plt.imshow(img_display)
        plt.axis('off')
        plt.show()

        img = image.load_img(file_name, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = img_array / 255.0
        img_batch = np.expand_dims(img_array, axis=0)

        predictions = model.predict(img_batch)
        predicted_index = np.argmax(predictions[0])
        confidence_score = np.max(predictions[0]) * 100

        predicted_disease = class_indices[predicted_index]

        print("=" * 40)
        if predicted_disease == 'Unknown_Crop':
            print(f"WARNING: Image recognized as an Unknown Crop with {confidence_score:.2f}% confidence.")
            print("Please upload a valid Tomato or Potato leaf.")
        else:
            # Format the result nicely (e.g. "Tomato: Early Blight")
            formatted_disease = predicted_disease.replace('___', ': ').replace('_', ' ').title()
            print(f"DIAGNOSIS:  {formatted_disease}")
            print(f"CONFIDENCE: {confidence_score:.2f}%")
        print("=" * 40)

# Run the widget
upload_and_predict()

Please upload a crop leaf image (Tomato or Potato):


Saving 110822-206-Tomato-blight.jpg to 110822-206-Tomato-blight.jpg
Loading model...


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'sish_nightshade_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)